# 🏥 Medical Q&A System — RAG + LLM

**Tech Stack:** Python | LangChain | FAISS | HuggingFace Embeddings | Groq LLM (Llama3)

**How it works:**
1. Load medical knowledge base and split into chunks
2. Convert chunks to vector embeddings using HuggingFace
3. Store embeddings in FAISS vector store
4. User asks a question → FAISS retrieves top 3 relevant chunks
5. Retrieved context + question → LLM generates grounded answer

**Diseases Covered:** Diabetes | Hypertension | CKD | Brain Stroke | Liver Disease | Lung Cancer | Heart Disease

## Step 1 — Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers python-dotenv

## Step 2 — Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate

print('✅ All libraries imported successfully')

## Step 3 — Set Groq API Key

> 🔑 Get your FREE key at: https://console.groq.com → Sign up → API Keys → Create Key

In [ ]:
# Paste your Groq API key here
os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'

print('✅ API Key set')

## Step 4 — Create Medical Knowledge Base

In [ ]:
medical_text = """
DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin or when the body cannot effectively use the insulin it produces.
Symptoms of diabetes include frequent urination, excessive thirst, unexplained weight loss, extreme fatigue, blurry vision, slow-healing cuts or bruises, and tingling or numbness in hands or feet.
Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. It requires daily insulin injections.
Type 2 diabetes is the most common form. It is managed through diet, exercise, and medication.
Risk factors for diabetes include obesity, physical inactivity, family history, age over 45, high blood pressure, and high cholesterol.
Complications include heart disease, kidney damage, eye damage, nerve damage, and foot problems.
Diagnosis is done through fasting blood glucose test, HbA1c test, or oral glucose tolerance test.

HYPERTENSION (HIGH BLOOD PRESSURE)
Hypertension is a condition where blood pressure is persistently elevated above 130/80 mmHg.
Most people have no symptoms, which is why it is called the silent killer.
Risk factors include obesity, high salt diet, smoking, excessive alcohol, stress, family history, and age.
Treatment includes lifestyle changes: reducing salt, regular exercise, weight loss, limiting alcohol, quitting smoking.
Medications include ACE inhibitors, calcium channel blockers, beta blockers, and diuretics.
Uncontrolled hypertension can lead to heart attack, stroke, kidney failure, and vision loss.

CHRONIC KIDNEY DISEASE (CKD)
Chronic kidney disease is the gradual loss of kidney function over time.
Main causes of CKD are diabetes and high blood pressure.
Symptoms include fatigue, swollen ankles, shortness of breath, blood in urine, foamy urine, and difficulty sleeping.
CKD is diagnosed through blood tests (creatinine, GFR) and urine tests (albumin).
Stages range from Stage 1 (mild) to Stage 5 (kidney failure requiring dialysis or transplant).
Treatment includes controlling blood sugar, managing blood pressure, low-protein diet, and avoiding NSAIDs.

BRAIN STROKE
A stroke occurs when blood supply to part of the brain is cut off or a blood vessel bursts.
Types include ischemic stroke (blockage, 87% of strokes) and hemorrhagic stroke (bleeding).
Risk factors include high blood pressure, diabetes, smoking, obesity, heart disease, high cholesterol, and age above 55.
Symptoms follow the FAST acronym: Face drooping, Arm weakness, Speech difficulty, Time to call emergency.
Treatment for ischemic stroke includes clot-busting drugs (tPA) given within 4.5 hours of onset.
Prevention includes controlling blood pressure, quitting smoking, regular exercise, healthy diet, and managing diabetes.

LIVER DISEASE
The liver performs over 500 functions including detoxification, protein synthesis, and digestion support.
Common liver diseases include fatty liver disease, hepatitis, cirrhosis, and liver cancer.
Symptoms include jaundice, abdominal pain, swelling, fatigue, nausea, and dark urine.
Diagnosis is done through liver function tests (LFTs), ultrasound, CT scan, and liver biopsy.
Treatment depends on the cause: antivirals for hepatitis, lifestyle changes for fatty liver, liver transplant for end-stage.

LUNG CANCER
Lung cancer is one of the most common and serious cancers worldwide.
Risk factors include smoking (85% of cases), secondhand smoke, radon gas, asbestos, air pollution, and family history.
Symptoms include persistent cough, coughing up blood, chest pain, shortness of breath, weight loss, and fatigue.
Treatment options include surgery, radiation, chemotherapy, targeted therapy, and immunotherapy.
Early detection significantly improves survival rates.

HEART DISEASE
Heart disease includes coronary artery disease, heart failure, and arrhythmias.
Symptoms include chest pain, shortness of breath, fatigue, and swelling in legs.
Risk factors include high blood pressure, high cholesterol, smoking, diabetes, obesity, and family history.
Heart attack occurs when blood flow to part of the heart is blocked.
Treatment includes medications, lifestyle changes, angioplasty, and bypass surgery.
"""

with open('medical_data.txt', 'w') as f:
    f.write(medical_text)

print('✅ Medical knowledge base saved')
print(f'Total characters: {len(medical_text)}')

## Step 5 — Load and Split into Chunks

In [ ]:
loader = TextLoader('medical_data.txt')
documents = loader.load()

print(f'✅ Loaded {len(documents)} document(s)')
print(f'Content length: {len(documents[0].page_content)} characters')

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f'✅ Total chunks created: {len(chunks)}')
print('\n--- Sample Chunk ---')
print(chunks[0].page_content)

## Step 6 — Create Vector Embeddings + Build FAISS Vector Store

In [ ]:
print('Loading embedding model... (first time may take 1-2 minutes)')

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('✅ Embedding model loaded')

In [ ]:
print('Building FAISS vector store...')

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local('faiss_index')

print(f'✅ Vector store built with {len(chunks)} vectors')
print('✅ Saved to faiss_index/')

## Step 7 — Test Retrieval (Without LLM)

Let's verify FAISS retrieves the correct context.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

test_query = 'What are the symptoms of diabetes?'
retrieved_docs = retriever.invoke(test_query)

print(f'Query: {test_query}')
print(f'\nTop {len(retrieved_docs)} retrieved chunks:\n')
for i, doc in enumerate(retrieved_docs):
    print(f'--- Chunk {i+1} ---')
    print(doc.page_content)
    print()

## Step 8 — Define Prompt Template (Prompt Engineering)

In [ ]:
PROMPT_TEMPLATE = """
You are a helpful medical assistant. Use the following medical context to answer 
the question clearly and accurately. If you don't know the answer from the context, 
say "I don't have enough information. Please consult a doctor."

Context:
{context}

Question: {question}

Answer:
"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=['context', 'question']
)

print('✅ Prompt template defined')
print('Input variables:', prompt.input_variables)

## Step 9 — Initialize Groq LLM (Llama3 — Free)

In [ ]:
llm = ChatGroq(
    model='llama3-8b-8192',
    api_key=os.environ.get('GROQ_API_KEY'),
    temperature=0.2
)

print('✅ Groq LLM (Llama3-8b) initialized')

## Step 10 — Build Full RAG Chain

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    chain_type_kwargs={'prompt': prompt},
    return_source_documents=True
)

print('✅ RAG Chain built!')
print('Pipeline: Question → FAISS Retrieval → Prompt → Llama3 → Answer')

## Step 11 — Ask Medical Questions

In [ ]:
def ask(question):
    print(f'\n🔍 Question: {question}')
    print('-' * 60)
    result = qa_chain.invoke({'query': question})
    print(f'💊 Answer:\n{result["result"]}')
    print('\n📄 Sources used:')
    for i, doc in enumerate(result['source_documents']):
        print(f'  Chunk {i+1}: {doc.page_content[:100]}...')
    return result['result']

print('✅ ask() function ready — run cells below to ask questions')

In [ ]:
ask('What are the symptoms of diabetes?')

In [ ]:
ask('How is hypertension treated?')

In [ ]:
ask('What causes chronic kidney disease?')

In [ ]:
ask('What are the risk factors for brain stroke?')

In [ ]:
ask('How is liver disease diagnosed?')

In [ ]:
ask('What are the symptoms of lung cancer?')

In [ ]:
# Try your own question!
ask('What are the complications of diabetes?')

## Step 12 — Reload Vector Store Next Time

Skip Steps 5–8 next time and just run this cell.

In [ ]:
# Uncomment and run this next time instead of rebuilding
# embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
# vectorstore = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)
# print('✅ Loaded from faiss_index/')

## Summary

| Step | What We Did |
|------|-------------|
| 1 | Installed libraries |
| 2 | Imported LangChain, FAISS, HuggingFace, Groq |
| 3 | Set Groq API key |
| 4 | Created medical knowledge base (7 diseases) |
| 5–6 | Loaded and split into 500-char chunks |
| 7–8 | Built HuggingFace embeddings + FAISS vector store |
| 9 | Tested retrieval |
| 10 | Defined prompt template (Prompt Engineering) |
| 11 | Initialized Groq Llama3 LLM |
| 12 | Built full RAG chain |
| 13 | Asked medical questions and got answers |

**Skills demonstrated:** RAG Architecture | LLM Integration | Prompt Engineering | FAISS Vector Store | LangChain | Healthcare AI | Flask Deployment